# Linear probe on frozen W2V + MERT embeddings

Najprostszy klasyfikator na **gotowych** pooled cechach `[1500, 2048]`:

```
utwór → mean po czasie → wektor [2048] (albo [1024] W2V / MERT) → LogisticRegression → bonafide vs spoof
```

**Cel:** sprawdzić, ile sygnału jest już w zamrożonych embeddingach, zanim ruszysz LoRA / re-preprocess.

Metryka: **EER** (niżej = lepiej), tak jak w BiMamba eval.

In [1]:
# odkomentuj jeśli brakuje sklearn
# !uv pip install scikit-learn

In [1]:
import json
import sys
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, precision_score, recall_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from torchmetrics.classification import EER
from tqdm.auto import tqdm

# repo root (notebook lives in notebooks/)
ROOT = Path.cwd()
if not (ROOT / "src").exists() and (ROOT.parent / "src").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

from src.utils.dataset import parse_split_label, parse_split_label_m6, parse_split_label_mom

SCRATCH = Path("/net/people/plgrid/plgjedrzejkusnierz/scratch/data")
SONICS_POOLED = SCRATCH / "Sonics/preprocessed_pooled_merged"
MOM_POOLED = SCRATCH / "MoM/preprocessed_pooled_fp16"
M6_POOLED = SCRATCH / "M6/preprocessed_pooled_fp16"

CACHE_DIR = ROOT / "notebooks" / "cache"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

# None = wszystkie pliki; np. 5000 = szybki smoke test
SUBSAMPLE_TRAIN = None
RANDOM_SEED = 42

In [2]:
def find_index(data_dir: Path) -> Path:
    matches = list(data_dir.rglob("index.json"))
    if not matches:
        raise FileNotFoundError(f"Brak index.json w {data_dir}")
    return matches[0]


def load_index_entries(
    data_dir: Path,
    parse_label,
    splits: set[str] | None = None,
) -> list[dict]:
    index_path = find_index(data_dir)
    with open(index_path, encoding="utf-8") as f:
        data = json.load(f)

    entries = []
    for item in data:
        stem = item.get("stem")
        pooled = item.get("pooled")
        if not stem or not pooled:
            continue
        split, label = parse_label(stem)
        if splits is not None and split not in splits:
            continue
        entries.append({"stem": stem, "split": split, "label": label, "path": pooled})
    return entries


def pooled_to_vector(path: str | Path, modality: str = "concat") -> np.ndarray:
    """[1500, 2048] fp16/fp32 → mean-pooled vector."""
    x = np.load(path).astype(np.float32)
    if modality == "concat":
        return x.mean(axis=0)
    if modality == "w2v":
        return x[:, :1024].mean(axis=0)
    if modality == "mert":
        return x[:, 1024:].mean(axis=0)
    raise ValueError(f"Unknown modality: {modality}")


def build_matrix(
    entries: list[dict],
    modality: str = "concat",
    cache_key: str | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    if cache_key is not None:
        cache_path = CACHE_DIR / f"{cache_key}.npz"
        if cache_path.exists():
            d = np.load(cache_path)
            return d["X"], d["y"]

    X = np.stack([pooled_to_vector(e["path"], modality) for e in tqdm(entries, desc=modality)])
    y = np.array([e["label"] for e in entries], dtype=np.int64)

    if cache_key is not None:
        np.savez(cache_path, X=X, y=y)
    return X, y


def compute_eer(probs: np.ndarray, y: np.ndarray) -> float:
    eer = EER(task="binary")
    scores = torch.tensor(probs, dtype=torch.float32)
    targets = torch.tensor(y, dtype=torch.int64)
    return float(eer(scores, targets))


def eval_split(name: str, clf: Pipeline, X: np.ndarray, y: np.ndarray) -> dict:
    probs = clf.predict_proba(X)[:, 1]
    preds = (probs >= 0.5).astype(int)
    return {
        "split": name,
        "n": len(y),
        "eer": compute_eer(probs, y),
        "f1": f1_score(y, preds, zero_division=0),
        "precision": precision_score(y, preds, zero_division=0),
        "recall": recall_score(y, preds, zero_division=0),
    }


def run_probe(
    train_entries,
    eval_sets: dict[str, tuple[np.ndarray, np.ndarray]],
    modality: str,
) -> pd.DataFrame:
    cache_key = f"sonics_{modality}_train" if SUBSAMPLE_TRAIN is None else None
    X_train, y_train = build_matrix(train_entries, modality, cache_key=cache_key)

    clf = Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
    ])
    clf.fit(X_train, y_train)

    rows = [eval_split(name, clf, X, y) for name, (X, y) in eval_sets.items()]
    out = pd.DataFrame(rows)
    out.insert(0, "modality", modality)
    return out

## Sonics (train → valid / test)

In [4]:
sonics = load_index_entries(SONICS_POOLED, parse_split_label, splits={"train", "valid", "test"})

by_split = {s: [e for e in sonics if e["split"] == s] for s in ("train", "valid", "test")}
for split, items in by_split.items():
    print(split, len(items), Counter(e["label"] for e in items))

train_entries = by_split["train"]
if SUBSAMPLE_TRAIN is not None and len(train_entries) > SUBSAMPLE_TRAIN:
    rng = np.random.default_rng(RANDOM_SEED)
    idx = rng.choice(len(train_entries), SUBSAMPLE_TRAIN, replace=False)
    train_entries = [train_entries[i] for i in idx]
    print(f"Subsample train → {len(train_entries)}")

results_sonics = []
for modality in ("concat", "w2v", "mert"):
    eval_sets = {}
    for split in ("valid", "test"):
        X, y = build_matrix(by_split[split], modality, cache_key=f"sonics_{modality}_{split}")
        eval_sets[split] = (X, y)
    results_sonics.append(run_probe(train_entries, eval_sets, modality))

pd.concat(results_sonics, ignore_index=True)

train 64050 Counter({1: 34023, 0: 30027})
valid 4261 Counter({1: 2273, 0: 1988})
test 24904 Counter({1: 12778, 0: 12126})


concat:   0%|          | 0/4261 [00:00<?, ?it/s]

concat:   0%|          | 0/24904 [00:00<?, ?it/s]

concat:   0%|          | 0/64050 [00:00<?, ?it/s]

w2v:   0%|          | 0/4261 [00:00<?, ?it/s]

...

w2v:   0%|          | 0/64050 [00:00<?, ?it/s]

mert:   0%|          | 0/4261 [00:00<?, ?it/s]

mert:   0%|          | 0/24904 [00:00<?, ?it/s]

mert:   0%|          | 0/64050 [00:00<?, ?it/s]

,modality,split,n,eer,f1,precision,recall
0,concat,valid,4261,0.002577,0.969875,0.999533,0.941927
1,concat,test,24904,0.004377,0.969560,0.999917,0.940992
2,w2v,valid,4261,0.006098,0.985733,0.999096,0.972723
3,w2v,test,24904,0.007669,0.983500,0.999596,0.967914
4,mert,valid,4261,0.001634,0.977288,0.999540,0.956005
5,mert,test,24904,0.003212,0.980819,0.999919,0.962435


## OOD — MoM & M6

Klasyfikator trenowany na Sonics train, testowany out-of-distribution (jak BiMamba eval).

In [5]:
ood_results = []

for name, pooled_dir, parse_fn in [
    ("MoM", MOM_POOLED, parse_split_label_mom),
    ("M6", M6_POOLED, parse_split_label_m6),
]:
    if not pooled_dir.exists():
        print(f"Pomijam {name}: brak {pooled_dir}")
        continue

    ood_entries = load_index_entries(pooled_dir, parse_fn)
    print(name, len(ood_entries), Counter(e["label"] for e in ood_entries))

    for modality in ("concat", "w2v", "mert"):
        X_ood, y_ood = build_matrix(ood_entries, modality, cache_key=f"{name}_{modality}")
        X_train, y_train = build_matrix(
            train_entries, modality, cache_key=f"sonics_{modality}_train"
        )

        clf = Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
        ])
        clf.fit(X_train, y_train)

        row = eval_split(name, clf, X_ood, y_ood)
        row["modality"] = modality
        ood_results.append(row)

pd.DataFrame(ood_results) if ood_results else print("Brak OOD datasetów na dysku.")

MoM 1098 Counter({1: 598, 0: 500})


concat:   0%|          | 0/1098 [00:00<?, ?it/s]

w2v:   0%|          | 0/1098 [00:00<?, ?it/s]

mert:   0%|          | 0/1098 [00:00<?, ?it/s]

M6 151 Counter({1: 80, 0: 71})


concat:   0%|          | 0/151 [00:00<?, ?it/s]

w2v:   0%|          | 0/151 [00:00<?, ?it/s]

mert:   0%|          | 0/151 [00:00<?, ?it/s]

,split,n,eer,f1,precision,recall,modality
0,MoM,1098,0.200334,0.776918,0.670717,0.923077,concat
1,MoM,1098,0.125709,0.847210,0.935354,0.774247,w2v
2,MoM,1098,0.202171,0.790393,0.699742,0.908027,mert
3,M6,151,0.437060,0.672986,0.541985,0.887500,concat
4,M6,151,0.225176,0.770270,0.838235,0.712500,w2v
5,M6,151,0.509771,0.679070,0.540741,0.912500,mert


## Jak czytać wyniki

| Porównanie | Co znaczy |
|------------|-----------|
| probe OOD vs BiMamba OOD (results.md) | czy Mamba dużo poprawia ponad mean-pool + LR |
| `w2v` vs `mert` vs `concat` | który encoder niesie sygnał → kandydat pod LoRA |
| valid vs test (Sonics) | czy probe w ogóle uczy się ID |

**Niski EER probe na Sonics + wysoki EER OOD** → frozen cechy słabo generalizują → LoRA na encoderze ma sens.

**Probe ≈ BiMamba OOD** → bottleneck raczej nie w encoderze; LoRA może dać mało.

## W2V probe — Sonics (valid / test) + OOD (MoM, M6)

Skupiony run **tylko na `w2v`** — jeden clf (`probe_w2v_clf`) trenowany na Sonics train, potem eval na:

- **Sonics:** valid, test (in-distribution)
- **OOD:** MoM, M6 (jak BiMamba eval)

Pipeline: `mean(W2V)` → `StandardScaler` (fit na train) → `LogisticRegression` (balanced).

**Wymaga:** komórki z importami i helperami (początek notebooka). Cache: `sonics_w2v_*`, `MoM_w2v`, `M6_w2v`.

In [3]:
MODALITY = "w2v"

if "train_entries" not in globals() or "by_split" not in globals():
    sonics = load_index_entries(SONICS_POOLED, parse_split_label, splits={"train", "valid", "test"})
    by_split = {s: [e for e in sonics if e["split"] == s] for s in ("train", "valid", "test")}
    train_entries = by_split["train"]
    if SUBSAMPLE_TRAIN is not None and len(train_entries) > SUBSAMPLE_TRAIN:
        rng = np.random.default_rng(RANDOM_SEED)
        idx = rng.choice(len(train_entries), SUBSAMPLE_TRAIN, replace=False)
        train_entries = [train_entries[i] for i in idx]
        print(f"Subsample train → {len(train_entries)}")
    print("Załadowano Sonics:", {s: len(by_split[s]) for s in by_split})

for split in ("train", "valid", "test"):
    print(f"{split:5} {len(by_split[split]):6} {Counter(e['label'] for e in by_split[split])}")

X_train_w2v, y_train_w2v = build_matrix(
    train_entries, MODALITY, cache_key=f"sonics_{MODALITY}_train"
)
probe_w2v_clf = Pipeline([
    ("scaler", StandardScaler()),
    ("lr", LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_SEED)),
])
probe_w2v_clf.fit(X_train_w2v, y_train_w2v)
print(f"Wytrenowano probe_w2v_clf na train (n={len(y_train_w2v)})")

w2v_rows = []
for split in ("valid", "test"):
    X, y = build_matrix(by_split[split], MODALITY, cache_key=f"sonics_{MODALITY}_{split}")
    w2v_rows.append(eval_split(split, probe_w2v_clf, X, y))

for ood_name, pooled_dir, parse_fn in [
    ("MoM", MOM_POOLED, parse_split_label_mom),
    ("M6", M6_POOLED, parse_split_label_m6),
]:
    if not pooled_dir.exists():
        print(f"Pomijam {ood_name}: brak {pooled_dir}")
        continue
    ood_entries = load_index_entries(pooled_dir, parse_fn)
    print(f"{ood_name} {len(ood_entries)} {Counter(e['label'] for e in ood_entries)}")
    X_ood, y_ood = build_matrix(ood_entries, MODALITY, cache_key=f"{ood_name}_{MODALITY}")
    w2v_rows.append(eval_split(ood_name, probe_w2v_clf, X_ood, y_ood))

probe_w2v_df = pd.DataFrame(w2v_rows)
probe_w2v_df.insert(0, "modality", MODALITY)

print("\n=== W2V linear probe (Sonics + OOD) ===")
probe_w2v_df

Załadowano Sonics: {'train': 128100, 'valid': 4261, 'test': 24904}
train 128100 Counter({1: 68046, 0: 60054})
valid   4261 Counter({1: 2273, 0: 1988})
test   24904 Counter({1: 12778, 0: 12126})
Wytrenowano probe_w2v_clf na train (n=128100)
MoM 1098 Counter({1: 598, 0: 500})
M6 151 Counter({1: 80, 0: 71})

=== W2V linear probe (Sonics + OOD) ===


,modality,split,n,eer,f1,precision,recall
0,w2v,valid,4261,0.021594,0.953585,0.998076,0.912890
1,w2v,test,24904,0.021683,0.951565,0.998710,0.908671
2,w2v,MoM,1098,0.088314,0.716418,0.988235,0.561873
3,w2v,M6,151,0.099296,0.655462,1.000000,0.487500
